# Weeks 43-44: paired challenge-set evaluation

Every model below is evaluated on `student_starter/challenge.json` **after** being trained or fitted on the training
split and with its threshold fixed on validation. Nothing here is tuned on the challenge set.

Metrics are reported separately for the **answerable** set and the **unanswerable** set:

- answerable set: answered rate (= answerable recall), and for span models SQuAD-v2 EM and token F1;
- unanswerable set: abstention rate (= unanswerable recall; for span models this equals the empty-prediction rate);
- pairs: share of pairs where the answerable record is answered and the unanswerable record is not
  (`pairs_correct`), and additionally with the exact span (`pairs_exact`).

The main table averages the per-language values over languages. The per-language table is for the appendix.

**Before running on Colab:** clone the repository as in your other notebooks (do not paste an access token into this
notebook), upload `data/` and the three QA checkpoint folders to `checkpoints/`, and run from the repository root.

In [18]:
try:
    from notebooks import nb_setup
except ImportError:
    import nb_setup

import json
import os
import platform
import time

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import f1_score

from src.data import PROJECT_ROOT, load_tokenised_data
from src.utils import enforce_reproducibility
from src.challenge_eval import (
    apply_threshold,
    classifier_result_rows,
    get_span_scores,
    qa_result_rows,
    span_labeller_result_rows,
    summarise_challenge,
)

SEED = 42
device = torch.device(
    "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
)
enforce_reproducibility(SEED)

CHECKPOINT_DIR = os.path.join(PROJECT_ROOT, "checkpoints")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Reproducibility record
print("python", platform.python_version(), "| torch", torch.__version__, "| device", device,
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")
all_results = {}
timings = {}

python 3.13.16 | torch 2.14.1+cu130 | device cuda | NVIDIA RTX PRO 6000 Blackwell Server Edition


## Data

In [19]:
import importlib.util

from src.preprocessing import tokenise

_spec = importlib.util.spec_from_file_location(
    "validate_challenge", os.path.join(PROJECT_ROOT, "student_starter", "validate_challenge.py"))
_validator = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_validator)
validate_records = _validator.validate_records

df_train, df_val = load_tokenised_data()

with open(os.path.join(PROJECT_ROOT, "student_starter", "challenge.json"), encoding="utf-8") as f:
    challenge_records = json.load(f)
print("validator errors:", validate_records(challenge_records, expected_per_member=10))

challenge_df = pd.DataFrame(challenge_records)
tokenised_q = [tokenise(r.question, r.lang) for r in challenge_df.itertuples()]
tokenised_c = [tokenise(r.context, "en") for r in challenge_df.itertuples()]
challenge_df["question_tokens"] = [t[0] for t in tokenised_q]
challenge_df["context_tokens"] = [t[0] for t in tokenised_c]
challenge_df["context_offsets"] = [t[1] for t in tokenised_c]

print(len(challenge_df), "records;", challenge_df["pair_id"].nunique(), "pairs;",
      challenge_df.groupby(["lang", "answerable"]).size().unstack().to_dict())

validator errors: []
30 records; 15 pairs; {False: {'ar': 4, 'ko': 4, 'te': 7}, True: {'ar': 4, 'ko': 4, 'te': 7}}


## Week 36 rule baseline and question translation

The NLLB translator is used for the rule baseline and for the translation extension's input, so both are produced
here and the translator is then freed.

In [20]:
from src.rule_based import rule_based_classifier
from src.translator import NLLBTranslator

translator = NLLBTranslator()

start = time.time()
rule_pred = [
    rule_based_classifier(r.question, r.context, r.lang, translator, threshold=0.3)
    for r in challenge_df.itertuples()
]
all_results["Rule (Week 36)"] = classifier_result_rows(challenge_df, rule_pred)

challenge_translated = challenge_df.copy()
for lang in ["ar", "ko", "te"]:
    idx = challenge_df.index[challenge_df["lang"] == lang]
    challenge_translated.loc[idx, "question"] = translator.translate_batch(
        challenge_df.loc[idx, "question"].tolist(), source_lang=lang, batch_size=8
    )
timings["translation + rule"] = time.time() - start

del translator
if torch.cuda.is_available():
    torch.cuda.empty_cache()
challenge_translated[["lang", "question"]]

config.json:   0%|          | 0.00/808 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 4.85MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.3MB            

...

Translating te:   0%|          | 0/1 [00:00<?, ?it/s]

Translating ar:   0%|          | 0/1 [00:00<?, ?it/s]

Translating ko:   0%|          | 0/1 [00:00<?, ?it/s]

Translating te:   0%|          | 0/2 [00:00<?, ?it/s]

,lang,question
0,ar,What's the original name of Easter Island?
1,ar,What's the Spanish name for Easter Island?
2,ko,What is the oldest culture known to have an ar...
3,ko,What is the oldest known culture for natural w...
4,te,What is the maximum distance between ALMA ante...
5,te,What is the minimum distance between ALMA ante...
6,ar,What is the name of the historic fort to visit...
7,ar,What is the name of the historic fort that can...
8,ko,What is the name of the world's largest salt f...
9,ko,What is the name of the second largest salt fl...


## Week 39 logistic regression (interpretable answerability baseline)

In [21]:
from src.answerability.logistic_regression import train_logistic_regression, predict_logistic_regression

start = time.time()
lr_model, lr_vectorizer = train_logistic_regression(df_train)
timings["logistic regression"] = time.time() - start

val_pred, _ = predict_logistic_regression(lr_model, lr_vectorizer, df_val)
print("validation macro-F1 (reported: 0.7236):",
      round(f1_score(df_val["answerable"].astype(int), val_pred, average="macro"), 4))

lr_pred, _ = predict_logistic_regression(lr_model, lr_vectorizer, challenge_df)
all_results["Logistic regression (Week 39)"] = classifier_result_rows(challenge_df, lr_pred)

validation macro-F1 (reported: 0.7236): 0.7236


## Week 39 pretrained GRU

The notebook that produced the reported numbers never saved the GRU, so it is retrained here with the same settings
and seed 42 (frozen XLM-R-base embeddings, hidden size 64, dropout 0.3, Adam 1e-3, up to 10 epochs, patience 3 on
validation macro-F1). Check that the validation macro-F1 below is close to the reported 0.8612; if it differs
(GPU non-determinism), report the retrained value for this table.

In [22]:
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import AutoModel, AutoTokenizer

from src.answerability.gru_dataset import PretrainedAnswerabilityDataset, make_collate_fn
from src.answerability.gru_model import (
    PretrainedGRUClassifier,
    evaluate_macro_f1,
    predict_answerability,
    train_one_epoch_answerability,
)

GRU_PATH = os.path.join(CHECKPOINT_DIR, "gru_answerability.pt")
xlmr_tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")
_xlmr = AutoModel.from_pretrained("xlm-roberta-base")
pretrained_embeddings = _xlmr.get_input_embeddings().weight.detach().cpu().clone()
del _xlmr

collate_fn = make_collate_fn(xlmr_tokenizer.pad_token_id)
val_loader = DataLoader(PretrainedAnswerabilityDataset(df_val, xlmr_tokenizer), batch_size=16,
                        shuffle=False, collate_fn=collate_fn)

enforce_reproducibility(SEED)
gru = PretrainedGRUClassifier(pretrained_embeddings, hidden_dim=64, dropout_prob=0.3,
                              padding_idx=xlmr_tokenizer.pad_token_id).to(device)

if os.path.exists(GRU_PATH):
    gru.load_state_dict(torch.load(GRU_PATH, map_location=device, weights_only=True))
    print("loaded", GRU_PATH)
else:
    train_loader = DataLoader(PretrainedAnswerabilityDataset(df_train, xlmr_tokenizer), batch_size=16,
                              shuffle=True, collate_fn=collate_fn)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, gru.parameters()), lr=1e-3, weight_decay=1e-4)
    best_f1, best_state, best_epoch, bad_epochs = -1, None, None, 0
    start = time.time()
    for epoch in range(10):
        loss = train_one_epoch_answerability(gru, train_loader, optimizer, criterion, device)
        f1 = evaluate_macro_f1(gru, val_loader, device)
        print(f"epoch {epoch + 1} | train loss {loss:.4f} | val macro-F1 {f1:.4f}")
        if f1 > best_f1:
            best_f1, best_epoch, bad_epochs = f1, epoch + 1, 0
            best_state = {k: v.detach().cpu().clone() for k, v in gru.state_dict().items()}
        else:
            bad_epochs += 1
            if bad_epochs >= 3:
                break
    timings["GRU training"] = time.time() - start
    gru.load_state_dict(best_state)
    torch.save(best_state, GRU_PATH)
    print("best epoch", best_epoch, "validation macro-F1", round(best_f1, 4))

gru.to(device)
challenge_loader = DataLoader(PretrainedAnswerabilityDataset(challenge_df, xlmr_tokenizer), batch_size=16,
                              shuffle=False, collate_fn=collate_fn)
gru_pred, gru_prob, _, _ = predict_answerability(gru, challenge_loader, device, threshold=0.5)
all_results["GRU (Week 39)"] = classifier_result_rows(challenge_df, gru_pred)

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


epoch 1 | train loss 0.1697 | val macro-F1 0.6986
epoch 2 | train loss 0.0939 | val macro-F1 0.6986
epoch 3 | train loss 0.0809 | val macro-F1 0.7632
epoch 4 | train loss 0.0734 | val macro-F1 0.8462
epoch 5 | train loss 0.0688 | val macro-F1 0.8746
epoch 6 | train loss 0.0558 | val macro-F1 0.8535
epoch 7 | train loss 0.0516 | val macro-F1 0.8298
epoch 8 | train loss 0.0397 | val macro-F1 0.7904
best epoch 5 validation macro-F1 0.8746


## Week 37 question-conditioned sequence labeller

Evaluated on its own terms: an unanswerable record is handled correctly when the predicted span is empty.

In [23]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

from src.bio_conversion import create_bio_labels
from src.features import featurise_df
from src.utils import split_by_lengths


def add_bio(df):
    labelled = df.apply(
        lambda r: create_bio_labels(r["answerable"], r["context"], r["context_tokens"],
                                    r["context_offsets"], r["answer_start"], r["answer"]),
        axis=1,
    )
    df = df.copy()
    df["bio_labels"], df["bio_alignment"] = zip(*labelled)
    return df


start = time.time()
span_train = add_bio(df_train)
X_dicts, y_train, _ = featurise_df(span_train)
span_vectorizer = DictVectorizer()
span_clf = LogisticRegression(max_iter=1000, class_weight={0: 1, 1: 3, 2: 3})
span_clf.fit(span_vectorizer.fit_transform(X_dicts), y_train)
timings["span labeller"] = time.time() - start

span_challenge = add_bio(challenge_df)
X_c, y_c, lengths_c = featurise_df(span_challenge)
pred_c = span_clf.predict(span_vectorizer.transform(X_c))
all_results["Span labeller (Week 37)"] = span_labeller_result_rows(
    challenge_df, split_by_lengths(pred_c, lengths_c), split_by_lengths(y_c, lengths_c)
)

## Weeks 40-41 integrated QA, translation extension and calibrated-abstention extension

Thresholds are frozen from validation:

- original QA: -1.779886 (token-F1-optimal threshold printed by notebook 7; update the constant below if you re-run notebook 7 and it prints a different value);
- translation extension: 0.717695 (notebook 8_1);
- calibrated abstention: thresholds fitted on all of validation from the exported scores of the original QA model
  (global macro-F1 threshold, and one threshold per language).
- zero-shot QA extension: 0.1306989336, selected using Arabic/Korean
  validation data in `8_zero_shot.ipynb`.

In [31]:
from transformers import AutoModelForQuestionAnswering
from src.calibration import GlobalThreshold, PerLanguageThreshold

QA_PATH = os.path.join(CHECKPOINT_DIR, "integrated_qa")
QA_TR_PATH = os.path.join(CHECKPOINT_DIR, "integrated_qa_translated")
ZERO_SHOT_PATH = os.path.join(CHECKPOINT_DIR, "zero_shot_telugu_qa")

for path in (QA_PATH, QA_TR_PATH, ZERO_SHOT_PATH):
    if not os.path.isdir(path):
        raise FileNotFoundError(f"{path} missing: copy the fine-tuned QA checkpoint folders here first.")

THRESHOLD_QA = -1.779886  # printed by notebook 7
THRESHOLD_QA_TRANSLATED = 0.717695
ZERO_SHOT_THRESHOLD = 0.1306989336

val_scores_path = os.path.join(PROJECT_ROOT, "data", "qa_val_predictions.csv")
print(val_scores_path)
if not os.path.exists(val_scores_path):
    raise FileNotFoundError(
        f"{val_scores_path} missing: run the export cell at the end of notebook 7 with the checkpoint in "
        f"{QA_PATH} so the calibration thresholds are fitted on that model's validation scores.")
val_scores = pd.read_csv(val_scores_path, keep_default_na=False)
y_v, s_v, l_v = val_scores["gold_answerable"].values, val_scores["no_answer_score"].values, val_scores["lang"].values
THRESHOLD_GLOBAL_MACRO = GlobalThreshold().fit(s_v, y_v, l_v).threshold
THRESHOLDS_PER_LANGUAGE = PerLanguageThreshold().fit(s_v, y_v, l_v).thresholds
print("frozen thresholds:", THRESHOLD_QA, THRESHOLD_QA_TRANSLATED, THRESHOLD_GLOBAL_MACRO, THRESHOLDS_PER_LANGUAGE, ZERO_SHOT_THRESHOLD)

langs = challenge_df["lang"].tolist()


tok = AutoTokenizer.from_pretrained(QA_PATH, use_fast=True)
qa = AutoModelForQuestionAnswering.from_pretrained(QA_PATH, attn_implementation="eager").to(device)
start = time.time()
scores = get_span_scores(qa, tok, challenge_df, device)
timings["QA inference"] = time.time() - start
all_results["QA (Weeks 40-41)"] = qa_result_rows(challenge_df, apply_threshold(scores, langs, THRESHOLD_QA))
all_results["QA + global macro-F1 threshold"] = qa_result_rows(
    challenge_df, apply_threshold(scores, langs, THRESHOLD_GLOBAL_MACRO))
all_results["QA + per-language thresholds (calibrated abstention)"] = qa_result_rows(
    challenge_df, apply_threshold(scores, langs, THRESHOLDS_PER_LANGUAGE))
del qa


tok_tr = AutoTokenizer.from_pretrained(QA_TR_PATH, use_fast=True)
qa_tr = AutoModelForQuestionAnswering.from_pretrained(QA_TR_PATH, attn_implementation="eager").to(device)
scores_tr = get_span_scores(qa_tr, tok_tr, challenge_translated, device)
all_results["QA + translation"] = qa_result_rows(
    challenge_df, apply_threshold(scores_tr, langs, THRESHOLD_QA_TRANSLATED))
del qa_tr

print("Loading zero-shot QA checkpoint:", ZERO_SHOT_PATH)

tok_zero = AutoTokenizer.from_pretrained(
    ZERO_SHOT_PATH,
    use_fast=True,
)

zero_model = AutoModelForQuestionAnswering.from_pretrained(
    ZERO_SHOT_PATH,
    attn_implementation="eager",
).to(device)

zero_model.eval()
start = time.time()

scores_zero = get_span_scores(
    zero_model,
    tok_zero,
    challenge_df,
    device,
)

timings["zero-shot QA inference"] = time.time() - start
all_results[
    "Zero-shot QA (Arabic/Korean fine-tuned)"
] = qa_result_rows(
    challenge_df,
    apply_threshold(
        scores_zero,
        langs,
        ZERO_SHOT_THRESHOLD,
    ),
)

print("Zero-shot QA challenge evaluation completed.")

del zero_model, tok_zero, scores_zero

/content/nlp-ku-project/data/qa_val_predictions.csv
frozen thresholds: -1.779886 0.717695 -0.157392515 {'ar': 0.506959325, 'ko': 0.89135415, 'te': -4.551388} 0.1306989336


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading zero-shot QA checkpoint: /content/nlp-ku-project/checkpoints/zero_shot_telugu_qa


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Zero-shot QA challenge evaluation completed.


## Results

In [32]:
main_table = summarise_challenge(all_results)
main_table

,model,n_answerable,answered,n_unanswerable,abstained,n_pairs,pairs_correct,em,f1,pairs_exact
0,Rule (Week 36),15.0,1.000,15.0,0.131,15.0,0.131,NaN,NaN,NaN
1,Logistic regression (Week 39),15.0,1.000,15.0,0.000,15.0,0.000,NaN,NaN,NaN
2,GRU (Week 39),15.0,1.000,15.0,0.000,15.0,0.000,NaN,NaN,NaN
3,Span labeller (Week 37),15.0,0.476,15.0,0.524,15.0,0.000,0.000,0.094,0.0
4,QA (Weeks 40-41),15.0,1.000,15.0,0.000,15.0,0.000,0.786,0.810,0.0
5,QA + global macro-F1 threshold,15.0,1.000,15.0,0.000,15.0,0.000,0.786,0.810,0.0
6,QA + per-language thresholds (calibrated abste...,15.0,1.000,15.0,0.000,15.0,0.000,0.786,0.810,0.0
7,QA + translation,15.0,1.000,15.0,0.000,15.0,0.000,0.952,0.984,0.0
8,Zero-shot QA (Arabic/Korean fine-tuned),15.0,1.000,15.0,0.000,15.0,0.000,0.702,0.736,0.0


In [33]:
per_language_table = summarise_challenge(all_results, by_language=True)
per_language_table

,model,lang,n_answerable,answered,n_unanswerable,abstained,n_pairs,pairs_correct,em,f1,pairs_exact
0,Rule (Week 36),ar,4,1.000,4,0.000,4,0.000,NaN,NaN,NaN
1,Rule (Week 36),ko,4,1.000,4,0.250,4,0.250,NaN,NaN,NaN
2,Rule (Week 36),te,7,1.000,7,0.143,7,0.143,NaN,NaN,NaN
3,Logistic regression (Week 39),ar,4,1.000,4,0.000,4,0.000,NaN,NaN,NaN
4,Logistic regression (Week 39),ko,4,1.000,4,0.000,4,0.000,NaN,NaN,NaN
5,Logistic regression (Week 39),te,7,1.000,7,0.000,7,0.000,NaN,NaN,NaN
6,GRU (Week 39),ar,4,1.000,4,0.000,4,0.000,NaN,NaN,NaN
7,GRU (Week 39),ko,4,1.000,4,0.000,4,0.000,NaN,NaN,NaN
8,GRU (Week 39),te,7,1.000,7,0.000,7,0.000,NaN,NaN,NaN
9,Span labeller (Week 37),ar,4,0.750,4,0.250,4,0.000,0.000,0.167,0.0


In [34]:
# Per-record predictions for the failure discussion
detail = pd.concat(
    [res.assign(model=name) for name, res in all_results.items()], ignore_index=True
)
detail["correct"] = detail["pred_answerable"] == detail["answerable"]
print(detail.pivot_table(index=["pair_id", "answerable"], columns="model", values="correct", aggfunc="first")
      .astype(int).to_string())

out_dir = os.path.join(PROJECT_ROOT, "data")
main_table.to_csv(os.path.join(out_dir, "challenge_results_main.csv"), index=False)
per_language_table.to_csv(os.path.join(out_dir, "challenge_results_per_language.csv"), index=False)
detail.to_csv(os.path.join(out_dir, "challenge_results_per_record.csv"), index=False)
print({k: round(v, 1) for k, v in timings.items()})

model                          GRU (Week 39)  Logistic regression (Week 39)  QA (Weeks 40-41)  QA + global macro-F1 threshold  QA + per-language thresholds (calibrated abstention)  QA + translation  Rule (Week 36)  Span labeller (Week 37)  Zero-shot QA (Arabic/Korean fine-tuned)
pair_id            answerable                                                                                                                                                                                                                                                          
alma               False                   0                              0                 0                               0                                                     0                 0               0                        0                                        0
                   True                    1                              1                 1                               1                                   